# protprep — Python API walkthrough

Protonating a protein at a given pH while **pinning the states of chosen
residues by hand**, then handing the structure to GROMACS `pdb2gmx`.

What this notebook covers:

1. [Setup](#setup)
2. [A first run](#first-run)
3. [Reading the report](#report)
4. [Standard pKa instead of PROPKA](#standard-pka)
5. [Hydrogens only on the pinned residues](#only-fixed-h)
6. [Capping the chain termini](#caps)
7. [Building the topology](#pdb2gmx)
8. [Spec files](#spec-files)
9. [When the force field cannot do it](#errors)

Requirements: `pip install protprep` (or `pip install git+https://github.com/Vetrov-Anton/protonate_pdb.git`)
and an installed GROMACS, which is where the force fields come from.

<a id="setup"></a>
## 1. Setup

In [ ]:
import os
import protprep
from protprep import Protonator, protonate, ForceFieldError
from protprep.ffdata import gromacs_top_dirs, resolve_ff_path

print("protprep", protprep.__version__)
print("GROMACS top dirs:", gromacs_top_dirs())

# a force field can be given as a path or as a bare name - it is looked up
# in the current directory and inside the GROMACS installation
FF = resolve_ff_path("amber99sb-ildn")
print("force field:", FF)

In [ ]:
# the example fragment shipped with the repository (chain A, residues 30-70)
PDB = "../examples/fragment.pdb"
if not os.path.exists(PDB):
    PDB = "examples/fragment.pdb"
WORK = "/tmp/protprep_demo"
print(PDB, "->", WORK)

<a id="first-run"></a>
## 2. A first run

Every configuration method returns the object itself, so the calls chain.
Nothing happens until `.run()`.

* `.fix("A", 31, "p")` — Asp31 protonated (`ASH`); `p`/`d`/`n`/`c` stand for
  protonated / deprotonated / neutral / charged
* `.fix("A:63:HIE")` — the same thing in single-token form, here choosing a
  histidine tautomer explicitly

In [ ]:
prot = (
    Protonator(PDB, ff=FF, ph=7.4)
    .fix("A", 31, "p")        # ASP -> ASH
    .fix("A:63:HIE")          # histidine tautomer
)
print(prot.describe())

In [ ]:
result = prot.run(f"{WORK}/run1")
print(result.summary())

<a id="report"></a>
## 3. Reading the report

`Result` carries the whole per-residue table, not just the file path.

In [ ]:
# every titratable residue and the state it ended up in
print("Asp31 ->", result.states[("A", 31)])
print("His63 ->", result.states[("A", 63)])

# only what actually moved relative to the input file
for r in result.changed():
    src = "PINNED" if r.forced else "propka"
    print(f"{r.chain}:{r.resid} {r.original} -> {r.final}  pKa={r.pka:.2f}  {src}")

In [ ]:
# as a DataFrame, if pandas is around
try:
    df = result.to_dataframe(only_changed=True)
    display(df)
except ImportError:
    print("pandas is not installed - use result.records()")
    print(result.records()[:2])

In [ ]:
# the reports are written next to the structure by default
print(sorted(os.listdir(f"{WORK}/run1")))
print()
print(open(f"{WORK}/run1/protonation_report.tsv").read()[:400])

<a id="standard-pka"></a>
## 4. Standard pKa instead of PROPKA

`standard_pka()` switches PROPKA off entirely: the pinned residues keep the
states you chose, and **everything else, termini included**, follows the
standard pKa values of free amino acids at the requested pH. Nothing is
computed from the environment, so it is also faster - on this 41-residue
fragment both are instant, but on a real structure (6CFO, four chains) it is
about 1 s against 11 s.

In [ ]:
import time

t0 = time.time()
propka_run = Protonator(PDB, ff=FF, ph=7.0).fix("A:31:p").run(f"{WORK}/propka")
t1 = time.time()
table_run = (
    Protonator(PDB, ff=FF, ph=7.0)
    .standard_pka()
    .fix("A:31:p")
    .run(f"{WORK}/standard")
)
t2 = time.time()

print(f"PROPKA   : {t1 - t0:.1f} s, {len(propka_run.changed())} residues shifted")
print(f"tabulated: {t2 - t1:.1f} s, {len(table_run.changed())} residues shifted")

# where the two disagree (the pinned residue is the same in both, by design)
diff = {
    key: (propka_run.states[key], table_run.states[key])
    for key in sorted(set(propka_run.states) & set(table_run.states))
    if propka_run.states[key] != table_run.states[key]
}
print()
print("states that differ:", diff or "none on this small fragment")

# the pKa values themselves always differ: local shifts vs the flat table
titratable = [r for r in propka_run.residues if r.pka is not None]
for r in titratable[:6]:
    print(f"{r.chain}:{r.resid} {r.original:>3}  propka={r.pka:6.2f}  model={r.model_pka:6.2f}")

<a id="only-fixed-h"></a>
## 5. Hydrogens only on the pinned residues

`only_fixed_hydrogens()` keeps hydrogens on the residues you pinned (and on the
caps) and lets `pdb2gmx` build the rest. Residue names are preserved, so the
protonation decisions survive — only the hydrogen coordinates are dropped.

In [ ]:
bare = (
    Protonator(PDB, ff=FF, ph=7.0)
    .standard_pka()
    .only_fixed_hydrogens()
    .fix("A:31:p")
    .run(f"{WORK}/bare")
)


def hydrogens_per_residue(path):
    counts = {}
    for line in open(path):
        if line.startswith("ATOM"):
            key = (line[21], int(line[22:26]), line[17:20].strip())
            counts[key] = counts.get(key, 0) + int(line[76:78].strip() == "H")
    return counts


counts = hydrogens_per_residue(bare.output_pdb)
with_h = {k: v for k, v in counts.items() if v}
print("residues that kept hydrogens:", with_h)
print("residues left bare:", sum(1 for v in counts.values() if not v))

<a id="caps"></a>
## 6. Capping the chain termini

`cap()` closes both ends at once; `nter()` / `cter()` set them separately.
Caps are built geometrically: trans peptide bond, standard lengths and angles,
methyl rotation picked to avoid contacts.

In [ ]:
capped = (
    Protonator(PDB, ff=FF, ph=7.0)
    .standard_pka()
    .cap("A", n="ACE", c="NME")     # or .nter("A", "ACE").cter("A", "NHE")
    .run(f"{WORK}/capped")
)
for note in capped.termini:
    print(note)

print()
print("".join(l for l in open(capped.output_pdb) if l[17:20] in ("ACE", "NME"))[:600])

<a id="pdb2gmx"></a>
## 7. Building the topology

`result.run_pdb2gmx()` runs GROMACS from the right directory and returns the
outcome, including the total charge it reported.

In [ ]:
run = capped.run_pdb2gmx()          # water="tip3p", gmx="gmx" by default
print("succeeded :", run.ok)
print("charge    :", run.total_charge)
print("gro / top :", run.gro, run.top)
if not run.ok:
    print("\n".join(run.tail(15)))

In [ ]:
# the same command, ready to paste into a shell
print(capped.pdb2gmx_cmd)

<a id="spec-files"></a>
## 8. Spec files

The same request can live in a text (or JSON) file and be shared with the CLI:
`protonate -f structure.pdb --spec spec.txt`.

In [ ]:
spec_text = '''
pH 7.0
pka standard
hydrogens fixed

A 31 p            # ASP -> ASH
A 63 HIE          # histidine tautomer
nter A ACE
cter A NME
'''
spec_path = f"{WORK}/spec.txt"
os.makedirs(WORK, exist_ok=True)
open(spec_path, "w").write(spec_text)

from_spec = Protonator.from_spec_file(PDB, spec_path, ff=FF)
print(from_spec.describe())
result_spec = from_spec.run(f"{WORK}/from_spec")
print()
print(result_spec.summary())

In [ ]:
# and the one-shot helper, when a whole object is overkill
quick = protonate(
    PDB, f"{WORK}/quick",
    fix=["A:31:p", ("A", 63, "HIE")],
    nter=[("A", "ACE")],
    ff=FF, ph=7.4, pka="standard",
)
print(quick.states[("A", 31)], quick.states[("A", 63)])

<a id="errors"></a>
## 9. When the force field cannot do it

protprep never edits the force field. If a required building block is missing,
it raises `ForceFieldError`, lists the reasons, and writes nothing at all.
A deprotonated tyrosine (`TYN`) is a good example: the GROMACS amber ports
neither list it in `residuetypes.dat` nor give it an integer charge.

In [ ]:
try:
    Protonator(PDB, ff=FF).fix("A", 35, "d").run(f"{WORK}/broken")
except ForceFieldError as err:
    print("refused, as expected:\n")
    for problem in err.problems:
        print(" *", problem)
    print("\ndirectory created:", os.path.exists(f"{WORK}/broken"))

## Cheat sheet

| call | what it does |
|------|--------------|
| `Protonator(pdb, ff=..., ph=...)` | start a run |
| `.fix("A", 145, "p")` / `.fix("A:145:p")` | pin one residue (`*` = all chains) |
| `.fix_many([...])` | pin several at once |
| `.nter("A", "ACE")` / `.cter("A", "NME")` | terminus state |
| `.cap("A")` | ACE + NME in one call |
| `.standard_pka()` | skip PROPKA, use tabulated pKa |
| `.only_fixed_hydrogens()` | H only on pinned residues and caps |
| `.run(outdir)` | do the work, return `Result` |
| `result.states` / `.changed()` / `.records()` / `.to_dataframe()` | the report |
| `result.summary()` | the text overview |
| `result.run_pdb2gmx()` | build the topology |
| `result.save_reports(dir)` | write the TSV/JSON reports |

States: `p` / `d` / `n` / `c`, or explicit names — `ASH`, `GLH`, `LYN`, `CYM`,
`CYX`, `TYN`, `ARN`, `HID`, `HIE`, `HIP`. For histidine `d` = `HID`,
`e` = `HIE`, `p` = `HIP`.